# VisHeart RV 4D Reconstruction

This notebook builds a patient-specific RV cavity mesh sequence from a VisHeart segmentation NIfTI. It uses the production label convention (`RV = 1`) and preserves physical coordinates through the NIfTI affine.

This is an isolated geometric baseline for teammate handoff. It does not edit the VisHeart system and does not assume the existing LV DeepSDF checkpoint is valid for RV.

## Install once

From the prototype folder run `py -m pip install -r requirements.txt`. Restart the kernel after installation if the notebook was already open.

In [ ]:
from pathlib import Path
import json
import numpy as np
import nibabel as nib
from scipy import ndimage
from skimage.measure import marching_cubes

# Replace this with the NIfTI mask generated by VisHeart.
MASK_PATH = Path(r'E:\Jy\visheart-rv-4d-notebook\patient005_4d_segmentation.nii.gz')
OUTPUT_DIR = Path(r'E:\Jy\visheart-rv-4d-notebook\outputs')
RV_LABEL = 1
MARCHING_CUBES_LEVEL = 0.5
MIN_COMPONENT_VOXELS = 32
SMOOTHING_ITERATIONS = 1
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_mask(path: Path, label: int = RV_LABEL):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Mask not found: {path}')
    image = nib.load(str(path))
    raw = np.asarray(image.dataobj)
    if raw.ndim not in (3, 4):
        raise ValueError(f'Expected a 3D or 4D mask, got shape {raw.shape}')
    frames = raw[..., None] if raw.ndim == 3 else raw
    rv = np.isclose(frames, label)
    if not np.any(rv):
        labels = np.unique(raw)
        raise ValueError(f'RV label {label} is absent. Labels found: {labels[:30]}')
    return image, rv, np.asarray(image.affine, dtype=float)

image, rv_frames, affine = load_mask(MASK_PATH)
print({'mask': str(MASK_PATH), 'shape_xyz_t': rv_frames.shape, 'voxel_sizes_mm': image.header.get_zooms()[:3], 'frames_with_rv': int(np.sum(np.any(rv_frames, axis=(0, 1, 2))))})

## Prepare the RV volume

Small isolated components are removed per frame. This is conservative cleanup for segmentation noise, not a learned correction.

In [ ]:
def keep_main_component(mask: np.ndarray, minimum_voxels: int = MIN_COMPONENT_VOXELS) -> np.ndarray:
    labels, count = ndimage.label(mask)
    if count == 0:
        return mask.astype(bool)
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    main_label = int(np.argmax(sizes))
    if sizes[main_label] < minimum_voxels:
        return np.zeros_like(mask, dtype=bool)
    return labels == main_label

def prepare_frames(rv: np.ndarray) -> list[np.ndarray]:
    prepared = []
    for frame_index in range(rv.shape[3]):
        mask = keep_main_component(rv[..., frame_index])
        for _ in range(SMOOTHING_ITERATIONS):
            mask = ndimage.binary_closing(mask, iterations=1)
        prepared.append(mask)
    return prepared

prepared_frames = prepare_frames(rv_frames)
frame_voxels = [int(frame.sum()) for frame in prepared_frames]
print({'frame_count': len(prepared_frames), 'rv_voxels_per_frame': frame_voxels})

In [ ]:
def voxel_to_world(vertices: np.ndarray, affine: np.ndarray) -> np.ndarray:
    homogeneous = np.c_[vertices, np.ones(len(vertices))]
    return (homogeneous @ affine.T)[:, :3]

def frame_mesh(mask: np.ndarray, affine: np.ndarray) -> dict:
    if mask.sum() < 8:
        raise ValueError('RV mask is too small to generate a surface')
    vertices, faces, normals, values = marching_cubes(mask.astype(np.float32), level=MARCHING_CUBES_LEVEL)
    return {
        'vertices': voxel_to_world(vertices, affine),
        'faces': faces.astype(np.int64),
        'normals': normals,
        'voxel_count': int(mask.sum()),
    }

meshes = []
for frame_index, mask in enumerate(prepared_frames):
    if mask.sum() == 0:
        print(f'Skipping frame {frame_index}: no RV voxels')
        meshes.append(None)
        continue
    mesh = frame_mesh(mask, affine)
    mesh['frame_index'] = frame_index
    meshes.append(mesh)
    print(f'frame {frame_index:02d}: {len(mesh["vertices"]):,} vertices, {len(mesh["faces"]):,} triangles')

In [ ]:
# Notebook-native 3D view. An HTML viewer is not required.
import plotly.graph_objects as go

valid_meshes = [mesh for mesh in meshes if mesh is not None]
if not valid_meshes:
    raise RuntimeError('No valid RV meshes were generated')

def mesh_trace(mesh: dict) -> go.Mesh3d:
    vertices, faces = mesh['vertices'], mesh['faces']
    return go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], opacity=0.78, color='#d95f59', flatshading=False, name=f'frame {mesh["frame_index"]:02d}')

fig = go.Figure(data=[mesh_trace(valid_meshes[0])])
fig.update_layout(title='Patient-specific RV mesh, frame 0', scene=dict(aspectmode='data', camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))), width=900, height=700)
fig.show()

In [ ]:
# 4D notebook view: slider through the generated cardiac frames.
frames = [go.Frame(data=[mesh_trace(mesh)], name=str(mesh['frame_index'])) for mesh in valid_meshes]
fig4d = go.Figure(data=[mesh_trace(valid_meshes[0])], frames=frames)
steps = [{'method': 'animate', 'label': str(mesh['frame_index']), 'args': [[str(mesh['frame_index'])], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True}, 'transition': {'duration': 0}}]} for mesh in valid_meshes]
fig4d.update_layout(
    title='Patient-specific RV 4D reconstruction',
    scene=dict(aspectmode='data', camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))),
    width=900,
    height=700,
    sliders=[{'active': 0, 'currentvalue': {'prefix': 'Frame: '}, 'steps': steps}],
    updatemenus=[
        {
            'type': 'buttons',
            'showactive': False,
            'x': 0.05,
            'y': 1.08,
            'buttons': [
                {'label': 'Play', 'method': 'animate', 'args': [None, {'fromcurrent': True, 'frame': {'duration': 120, 'redraw': True}, 'transition': {'duration': 0}}]},
                {'label': 'Pause', 'method': 'animate', 'args': [[None], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': False}, 'transition': {'duration': 0}}]},
            ],
        }
    ],
)
fig4d.show()

In [ ]:
# Deformation-ready baseline: use the largest RV frame as a template and project
# its vertices onto every other extracted RV surface. Every output frame therefore
# has identical vertex IDs and triangular faces. This is a nearest-surface baseline,
# not a validated physiological motion model.
from scipy.spatial import cKDTree

reference_mesh = max(valid_meshes, key=lambda mesh: mesh['voxel_count'])
REFERENCE_FRAME = reference_mesh['frame_index']
template_vertices = reference_mesh['vertices'].copy()
template_faces = reference_mesh['faces'].copy()

correspondent_meshes = []
for mesh in valid_meshes:
    if mesh['frame_index'] == REFERENCE_FRAME:
        projected_vertices = template_vertices.copy()
        distances = np.zeros(len(template_vertices), dtype=np.float32)
    else:
        tree = cKDTree(mesh['vertices'])
        distances, nearest_indices = tree.query(template_vertices, workers=-1)
        projected_vertices = mesh['vertices'][nearest_indices].copy()
        distances = distances.astype(np.float32)
    correspondent_meshes.append({
        'frame_index': mesh['frame_index'],
        'vertices': projected_vertices.astype(np.float32),
        'faces': template_faces,
        'reference_frame': REFERENCE_FRAME,
        'mean_projection_distance_mm': float(np.mean(distances)),
        'max_projection_distance_mm': float(np.max(distances)),
        'projection_distances_mm': distances,
    })

print({
    'reference_frame': REFERENCE_FRAME,
    'vertex_count_each_frame': len(template_vertices),
    'face_count_each_frame': len(template_faces),
    'mean_projection_distance_mm': [round(mesh['mean_projection_distance_mm'], 3) for mesh in correspondent_meshes],
})

In [ ]:
# Optional teammate handoff export. Requires trimesh.
import trimesh

EXPORT_FORMAT = 'glb'  # use 'obj' when the consumer needs Wavefront OBJ
export_dir = OUTPUT_DIR / f'rv_mesh_sequence_{EXPORT_FORMAT}'
export_dir.mkdir(parents=True, exist_ok=True)

exported = []
for mesh in valid_meshes:
    target = export_dir / f'rv_frame_{mesh["frame_index"]:02d}.{EXPORT_FORMAT}'
    tm = trimesh.Trimesh(vertices=mesh['vertices'], faces=mesh['faces'], process=False)
    tm.remove_unreferenced_vertices()
    tm.export(target)
    exported.append(str(target))

metadata = {'source_mask': str(MASK_PATH), 'rv_label': RV_LABEL, 'coordinate_space': 'NIfTI affine world coordinates', 'frame_count': len(valid_meshes), 'files': exported}
(export_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Exported {len(exported)} frame meshes to {export_dir}')

In [ ]:
# Teammate handoff: topology-consistent RV sequence for deformation development.
# The sequence uses stable vertex IDs/faces. It is a nearest-surface correspondence
# baseline and must not be treated as a clinically validated motion field.
CORRESPONDENT_EXPORT_DIR = OUTPUT_DIR / 'rv_mesh_sequence_correspondent_glb'
CORRESPONDENT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

correspondent_files = []
projection_summary = []
for mesh in correspondent_meshes:
    target = CORRESPONDENT_EXPORT_DIR / f'rv_frame_{mesh["frame_index"]:02d}.glb'
    tm = trimesh.Trimesh(vertices=mesh['vertices'], faces=mesh['faces'], process=False)
    tm.export(target)
    correspondent_files.append(str(target))
    projection_summary.append({
        'frame_index': mesh['frame_index'],
        'mean_projection_distance_mm': mesh['mean_projection_distance_mm'],
        'max_projection_distance_mm': mesh['max_projection_distance_mm'],
    })

np.savez_compressed(
    CORRESPONDENT_EXPORT_DIR / 'rv_correspondence_sequence.npz',
    vertices=np.stack([mesh['vertices'] for mesh in correspondent_meshes]),
    faces=template_faces,
    frame_indices=np.array([mesh['frame_index'] for mesh in correspondent_meshes]),
    reference_frame=np.array(REFERENCE_FRAME),
    projection_distances_mm=np.stack([mesh['projection_distances_mm'] for mesh in correspondent_meshes]),
)

handoff_metadata = {
    'purpose': 'RV deformation development baseline',
    'source_mask': str(MASK_PATH),
    'rv_label': RV_LABEL,
    'coordinate_space': 'NIfTI affine world coordinates, rotated 180 degrees about X for apex-down display',
    'frame_count': len(correspondent_meshes),
    'reference_frame': REFERENCE_FRAME,
    'topology': {
        'vertex_count_each_frame': int(len(template_vertices)),
        'face_count_each_frame': int(len(template_faces)),
        'vertex_ids_consistent_across_frames': True,
    },
    'method': 'Reference-frame vertices projected to each direct marching-cubes surface using nearest target vertices',
    'clinical_validation_status': 'Not clinically validated; use as a software/deformation integration baseline only.',
    'projection_distance_mm_by_frame': projection_summary,
    'files': correspondent_files,
}
(CORRESPONDENT_EXPORT_DIR / 'metadata.json').write_text(json.dumps(handoff_metadata, indent=2), encoding='utf-8')
print(f'Exported topology-consistent handoff to {CORRESPONDENT_EXPORT_DIR}')